In [ ]:
#setup for processed data
#Note: use for BinaryArray data produced from Entrainment_Preprocessing.ipynb
def GetProcessedString(PROCESSING=False):
    if PROCESSING==True:
        Processed_string="PROCESSED_"
    else:
        Processed_string=""
    return Processed_string

PROCESSING=False 
# PROCESSING=True #set to True if using Turbulence-Removed Binary Arrays
Processed_string = GetProcessedString(PROCESSING=PROCESSING)

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from FUNCTIONS_Variable_Calculation import *

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",
                                dtype='int32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Load Data Functions
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z', 'Y', 'X']
    dataDictionary = MakeDataDictionary(variableNames,t)
    Z,Y,X = (dataDictionary[k] for k in variableNames)
    return Z,Y,X

def GetVariableData(t):    
    variableNames = ['QV']
    dataDictionary = MakeDataDictionary(variableNames,t)
    QV, = (dataDictionary[k] for k in variableNames)
    return QV
def GetAData(t):  
    variableNames = [f'{Processed_string}A_g', f'{Processed_string}A_c']

    dataDictionary = MakeDataDictionary(variableNames,t, printstatement=False)
    A_g,A_c = (dataDictionary[k] for k in variableNames)
    return A_g,A_c

def GetAData_Eulerian(t):
    variableNames = [f'A_g', f'A_c']
    [A_g,A_c] = [CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName) for variableName in variableNames]
    return A_g,A_c
def GetVariableData_Eulerian(t):
    variableNames = [f'qv']
    [QV]=[CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName) for variableName in variableNames]
    return QV

def GetQCQI(t,
            varName='QCQI'):    
    variableNames = [varName]
    dataDictionary = MakeDataDictionary(variableNames,t)
    QCQI, = (dataDictionary[k] for k in variableNames)
    return QCQI

In [ ]:
#Calculation Functions
def CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, A1,A2, A1_Prior,A2_Prior,
                         QV,QV_Prior,QV_Prior_Eulerian,A2_Prior_Eulerian,
                         QCQI,QCQI_Prior,
                         isRemoveCondOrEvapEvents=True):
    """
    Function to compute 3D entrainment and update result array based on provided inputs.
    
    Returns a 3D (t,z) array containing the sum of the D array representing entrained parcels, by 1, and detrained parcels, by -1.
    The finally array is then ordered by the appropiate index using the np.add.at function
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.
    - Y: The (t,p) Lagrangian y index array.
    - X: The (t,p) Lagrangian x index array.

    """
    #Calculation for Entrainment and Detrainment
    DMatrix_Entrainment = SubtractA(A2,A2_Prior)
    DMatrix_Detrainment = DMatrix_Entrainment.copy()

    # Update D for entrainment/detrainment
    DMatrix_Entrainment[DMatrix_Entrainment < 0] = 0
    DMatrix_Detrainment[DMatrix_Detrainment > 0] = 0

    # Require entraining parcel to be outside cloud at previous timestep #*testing
    # D_raw = SubtractA(A2, A2_Prior)  
    # entrainRaw = D_raw > 0
    # entrainAfterQCQI = entrainRaw & (QCQI_Prior <= 1e-6)
    # print("entrainment raw:", np.sum(entrainRaw))
    # print("entrainment kept:", np.sum(entrainAfterQCQI))
    # print("entrainment removed:", np.sum(entrainRaw & (QCQI_Prior > 1e-6)))
    DMatrix_Entrainment[QCQI_Prior > 1e-6] = 0
    DMatrix_Detrainment[QCQI > 1e-6] = 0

    ##########
    #General <==> Cloudy Updraft-Transfer Entrainment
    ########## c to g AND g to c
    SMatrix_Entrainment = ((A1_Prior == 1) & (A2 == 1) & (A2_Prior == 0)).astype(np.int8) #A1==>A2 entrainment #Note: last conditional removes double-counting where A1 = A2 = 1)
    SMatrix_Detrainment = SMatrix_Entrainment.copy()
    ##########

    # Remove in-situ condensation/evaporation events
    if isRemoveCondOrEvapEvents:
        DMatrix_Entrainment=RemoveCondOrEvapEvents(DMatrix_Entrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        DMatrix_Detrainment=RemoveCondOrEvapEvents(DMatrix_Detrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        SMatrix_Entrainment=RemoveCondOrEvapEvents(SMatrix_Entrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
        SMatrix_Detrainment=RemoveCondOrEvapEvents(SMatrix_Detrainment,Z,Y,X,Z_Prior,Y_Prior,X_Prior)
    
    # Initialize time and vertical dimension arrays
    Nz = ModelData.Nzh; Ny = ModelData.Nyh; Nx = ModelData.Nxh
    
    # Initialize result array
    Entrainment = np.zeros((Nz, Ny, Nx),dtype="int32"); Detrainment = Entrainment.copy()
    TransferEntrainment = Entrainment.copy(); TransferDetrainment = Entrainment.copy()
    
    # Initializing dry/moist entrainment arrays
    Entrainment_Dry = Entrainment.copy(); Entrainment_Moist = Entrainment.copy()

    if t==0:
        return (Entrainment,Detrainment,TransferEntrainment,TransferDetrainment,
                Entrainment_Dry,Entrainment_Moist)
    else:
        # Use np.add.at to accumulate values in the result array
        np.add.at(Entrainment, (Z, Y, X), DMatrix_Entrainment)
        np.add.at(Detrainment, (Z_Prior, Y_Prior, X_Prior), DMatrix_Detrainment)
        np.add.at(TransferEntrainment, (Z, Y, X), SMatrix_Entrainment)
        np.add.at(TransferDetrainment, (Z_Prior, Y_Prior, X_Prior), SMatrix_Detrainment)

        # Calculating dry/moist entrainment
        qv_u=QV
        # qv_u = GetQV_Updraft(QV_Prior_Eulerian, 
        #                      Z,Y,X, 
        #                      A2_Prior_Eulerian,DMatrix_Entrainment)
        [isDry,isMoist]  = CompareQuAndQe(qv_e=QV_Prior,qv_u=qv_u)

        dryMask = DMatrix_Entrainment * isDry
        moistMask = DMatrix_Entrainment * isMoist
        np.add.at(Entrainment_Dry, (Z, Y, X), dryMask) #dry entrainment
        np.add.at(Entrainment_Moist, (Z, Y, X),  moistMask) #moist entrainment

        
        return (Entrainment,Detrainment,TransferEntrainment,TransferDetrainment,
                Entrainment_Dry,Entrainment_Moist,
                dryMask.astype(bool),moistMask.astype(bool))

def SubtractA(A,A_Prior):
    D = np.zeros_like(A,dtype=np.int8)
    D = A*1 - A_Prior*1
    return D

def RemoveCondOrEvapEvents(array,
                           Z,Y,X,Z_Prior,Y_Prior,X_Prior):
    ParcelMoved = (Z != Z_Prior) | (Y != Y_Prior) | (X != X_Prior)
    # ParcelMoved = (Z == Z_Prior) & ((Y != Y_Prior) | (X != X_Prior)) #*testing only lateral entrainment
    array[~ParcelMoved]=0
    return array

In [ ]:
def CompareQuAndQe(qv_e,qv_u):
    validMask = np.isfinite(qv_u)

    isDry = (qv_e < qv_u) & validMask
    isMoist = (qv_e > qv_u) & validMask
    return isDry,isMoist

from scipy.spatial import cKDTree
def GetQV_Updraft(QV_Prior_Eulerian,
                  Z,Y,X, 
                  A_Prior_Eulerian,DMatrix_Entrainment):
    qv_u = np.full(DMatrix_Entrainment.shape, np.nan, dtype=float)

    entrainParcelMask = DMatrix_Entrainment > 0
    if not np.any(entrainParcelMask): return qv_u

    cloudyGridMask = A_Prior_Eulerian == 1
    cloudy_pos = np.column_stack(np.where(cloudyGridMask))
    if cloudy_pos.shape[0] == 0: return qv_u

    entrain_pos = np.column_stack([Z[entrainParcelMask],
                                   Y[entrainParcelMask],X[entrainParcelMask]])

    tree = cKDTree(cloudy_pos)
    _, nearest = tree.query(entrain_pos, k=1)

    nearest_pos = cloudy_pos[nearest]

    qv_u[entrainParcelMask] = QV_Prior_Eulerian[nearest_pos[:, 0],
                                                nearest_pos[:, 1],
                                                nearest_pos[:, 2]]

    return qv_u

In [ ]:
#Running Function
def RunCalculation(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, A_g,A_c, A_g_Prior,A_c_Prior,
                   QV,QV_Prior,QV_Prior_Eulerian,
                   A_g_Prior_Eulerian,A_c_Prior_Eulerian,
                   QCQI,QCQI_Prior,
                   Processed_string,isRemoveCondOrEvapEvents=True): 

    [Entrainment_g, Detrainment_g,
     TransferEntrainment_g,TransferDetrainment_c,
     Entrainment_Dry_g,Entrainment_Moist_g,
     dryMask_g,moistMask_g] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior,
                                                                   A1=A_c,A2=A_g,
                                                                   A1_Prior=A_c_Prior,A2_Prior=A_g_Prior,
                                                                   QV=QV,QV_Prior=QV_Prior,
                                                                   QV_Prior_Eulerian=QV_Prior_Eulerian,
                                                                   A2_Prior_Eulerian=A_g_Prior_Eulerian,
                                                                   QCQI=QCQI,QCQI_Prior=QCQI_Prior,
                                                                   isRemoveCondOrEvapEvents\
                                                                   =isRemoveCondOrEvapEvents)

    [Entrainment_c, Detrainment_c,
     TransferEntrainment_c,TransferDetrainment_g,
     Entrainment_Dry_c,Entrainment_Moist_c,
     dryMask_c,moistMask_c] = CalculateEntrainment(t, Z,Y,X,Z_Prior,Y_Prior,X_Prior, 
                                                                   A1=A_g,A2=A_c, 
                                                                   A1_Prior=A_g_Prior,A2_Prior=A_c_Prior,
                                                                   QV=QV,QV_Prior=QV_Prior,
                                                                   QV_Prior_Eulerian=QV_Prior_Eulerian,
                                                                   A2_Prior_Eulerian=A_c_Prior_Eulerian,
                                                                   QCQI=QCQI,QCQI_Prior=QCQI_Prior,
                                                                   isRemoveCondOrEvapEvents\
                                                                   =isRemoveCondOrEvapEvents)

    outputDictionary_Entrainment = {
        f"{Processed_string}Entrainment_g": Entrainment_g,
        f"{Processed_string}Entrainment_c": Entrainment_c,
        f"{Processed_string}TransferEntrainment_g": TransferEntrainment_g, #c to g
        f"{Processed_string}TransferEntrainment_c": TransferEntrainment_c, #g to c
        
        f"{Processed_string}Entrainment_Dry_g": Entrainment_Dry_g,
        f"{Processed_string}Entrainment_Moist_g": Entrainment_Moist_g,
        f"{Processed_string}Entrainment_Dry_c": Entrainment_Dry_c,
        f"{Processed_string}Entrainment_Moist_c": Entrainment_Moist_c,
        }
    
    outputDictionary_Detrainment = {
        f"{Processed_string}Detrainment_g": Detrainment_g,
        f"{Processed_string}Detrainment_c": Detrainment_c,
        f"{Processed_string}TransferDetrainment_g": TransferDetrainment_g, #from g to c
        f"{Processed_string}TransferDetrainment_c": TransferDetrainment_c, #from c to g
        }

    outputDictionary_Entrainment.update({
        f"{Processed_string}dryMask_g": dryMask_g,
        f"{Processed_string}moistMask_g": moistMask_g,
        f"{Processed_string}dryMask_c": dryMask_c,
        f"{Processed_string}moistMask_c": moistMask_c,
        })

    return outputDictionary_Entrainment, outputDictionary_Detrainment

In [ ]:
####################################
#CALCULATING ENTRAINMENT CONSTANT

In [ ]:
#Functions

#constants
def GetConstants():
    Cp=1004 #Jkg-1K-1
    Cv=717 #Jkg-1K-1
    Rd=Cp-Cv #Jkg-1K-1
    eps=0.608
    return Cp,Cv,Rd,eps

def GetNumerics():
    Np=ModelData.Np #number of lagrangian parcles

    # Lx=(ModelData.xf[-1]-ModelData.xf[0])*1000 #x length (m)
    # Ly=(ModelData.yf[-1]-ModelData.yf[0])*1000 #y length (m)
    dt=ModelData.dt #seconds
    dz=ModelData.dz #meters
    dy=ModelData.dy #meters
    dx=ModelData.dx #meters
    
    # zfs=ModelData.zf*1000
    # dz = np.diff(zfs)

    return Np, dt, dz,dy,dx

def zf(k):
    out=ModelData.zf[k]*1000
    return out

#calculation functions
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out
    
def rho(x,y,z,rho_data_t):
    out=rho_data_t[z,y,x]
    return out
    
def Calculate_dm(t, dy,dx, Np):
    timeString = ModelData.timeStrings[t]
    rho_data_t = CallVariable(ModelData, DataManager, timeString, "rho")
    
    #calculating
    m=0
    for k in range(ModelData.Nzh):
        dz=(zf(k+1)-zf(k))
        for j in range(ModelData.Nyh):
            for i in range(ModelData.Nxh):
                rho_out=rho(i,j,k,rho_data_t)
                m+=rho_out*dz

    #multiplying by integration differentials
    dm = m*dx*dy/Np
    return dm

#calculating constant
def ComputeEntrainmentConstant(t=0):
    #constants
    [Cp,Cv,Rd,eps] = GetConstants()
    [Np, dt, dz,dy,dx] = GetNumerics()

    #calculation
    dm = Calculate_dm(t, dy,dx, Np); #print(dm)
    divisor=dt*dz*dy*dx
    entrainmentConstant = dm/divisor

    outputDictionary = {"entrainmentConstant": entrainmentConstant}
    return outputDictionary

#calculating constant
def ComputeMassFluxConstant(t=0):
    #constants
    # [Cp,Cv,Rd,eps] = GetConstants()
    [Np, dt, dz,dy,dx] = GetNumerics()

    #calculation
    dm = Calculate_dm(t, dy,dx, Np); print(dm)
    divisor=dz*dy*dx
    massFluxConstant = dm/divisor

    outputDictionary = {"massFluxConstant": massFluxConstant}
    return outputDictionary

In [ ]:
#Calculating
try:
    DataManager_Entrainment = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="EntrainmentCalculation", dataName="EntrainmentCalculation",dtype='int32',verbose=False)

    entrainmentConstant = DataManager_Entrainment.LoadCalculations(DataManager_Entrainment.outputDataDirectory,
                                                                   dataName="EntrainmentConstant",
                                                                   verbose=False,)["entrainmentConstant"]
except:
    #calculating
    outputDictionary_EntrainmentConstant = ComputeEntrainmentConstant()
    entrainmentConstant = outputDictionary_EntrainmentConstant["entrainmentConstant"]
    
    #saving
    DataManager.SaveCalculations(DataManager.outputDataDirectory, outputDictionary_EntrainmentConstant, dataName="EntrainmentConstant",dtype="float32")

    #plotting
    plt.plot(entrainmentConstant,ModelData.zh,color='black')
    plt.ylabel("z (km)")
    plt.xlabel("Entrainment Constant (kg/m^3/s)")
    plt.title("Plotting Vertical Profile of Entrainment Constant\n (due to stretched z-grid)");

# def ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment):

#     for key, value in outputDictionary_Entrainment.items():
#         outputDictionary_Entrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

#     for key, value in outputDictionary_Detrainment.items():
#         outputDictionary_Detrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

def ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment):
    for key, value in outputDictionary_Entrainment.items():
        if "dryMask" in key or "moistMask" in key:
            continue
        outputDictionary_Entrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]
    for key, value in outputDictionary_Detrainment.items():
        if "dryMask" in key or "moistMask" in key:
            continue
        outputDictionary_Detrainment[key] = value * entrainmentConstant[:, np.newaxis, np.newaxis]

In [ ]:
####################################
#TESTING

In [ ]:
t=np.abs(ModelData.time_hrs-15.5).argmin()
print(f'{ModelData.time_hrs[t]} LT')

#loading data
[Z,Y,X] = GetSpatialData(t); [Z_Prior,Y_Prior,X_Prior] = GetSpatialData(t-1)
[A_g,A_c] = GetAData(t); [A_g_Prior,A_c_Prior] = GetAData(t-1) 
[A_g_Prior_Eulerian,A_c_Prior_Eulerian]=GetAData_Eulerian(t-1)
QCQI = GetQCQI(t); QCQI_Prior = GetQCQI(t-1)

#loading moist/dry entrainment calculation variables
QV = GetVariableData(t); QV_Prior = GetVariableData(t-1)
QV_Prior_Eulerian=GetVariableData_Eulerian(t-1)

#calculating variables
[outputDictionary_Entrainment,outputDictionary_Detrainment] = RunCalculation(t,
                                                                             Z,Y,X,Z_Prior,Y_Prior,X_Prior, A_g,A_c,
                                                                             A_g_Prior,A_c_Prior,
                                                                             QV,QV_Prior,QV_Prior_Eulerian,
                                                                             A_g_Prior_Eulerian,A_c_Prior_Eulerian,
                                                                             QCQI,QCQI_Prior,
                                                                             Processed_string)

#applying entrainment (density rate) constant
ApplyEntrainmentConstant(outputDictionary_Entrainment, outputDictionary_Detrainment)

In [ ]:
#Test Plot

#IMPORT FUNCTIONS
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

def TestPlot_Average(e,e_dry,e_moist,updraftType,
             cmap='turbo',numLevels=101):

    numLevels=100
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    
    plotX = ModelData.xh-ModelData.xh[0]
    plotY = ModelData.zh
    
    data = [(e,       'Total'),
            (e_dry,   'Dry'),
            (e_moist, 'Moist'),]
    
    # shared color scale across all three panels so they're visually comparable
    vmax = max(np.nanmax(np.mean(d, axis=1)) for d, _ in data); vmin = min(np.nanmin(np.mean(d, axis=1)) for d, _ in data)
    levels = np.linspace(vmin, vmax, numLevels)
    
    for ax, (d, title) in zip(axes, data):
        plotZ = np.mean(d, axis=1)
        cf = ax.contourf(plotX, plotY, plotZ, levels=levels, cmap=cmap)
        ax.set_title(title);

        yLims=(0,1.5) if updraftType=='g' else (1.5,4)
        xLims=(150,None)
        
        ax.set_ylim(yLims)
        ax.set_xlim(xLims)
        ax.set_xlabel('x (km)')
    
    axes[0].set_ylabel('z (km)')
    
    cbar=fig.colorbar(cf, ax=axes, shrink=0.8, label='Entrainment')
    apply_scientific_notation_colorbar([cbar])


updraftType='g'
updraftType='c'

e=outputDictionary_Entrainment[f'{Processed_string}Entrainment_{updraftType}']
e_dry=outputDictionary_Entrainment[f'{Processed_string}Entrainment_Dry_{updraftType}']
e_moist=outputDictionary_Entrainment[f'{Processed_string}Entrainment_Moist_{updraftType}']
TestPlot_Average(e,e_dry,e_moist,updraftType)

In [ ]:
####################################
#EXPLAINING WHY "MOIST" ENTRAINMENT SO PREVALENT

In [ ]:
# ============================================================
# CallLagrangianArray_Function
# ============================================================

import os

def CallLagrangianArray(ModelData, DataManager, timeString, variableName, 
                        printstatement=False,loadData=True):
    Processed_string = "PROCESSED_" if "PROCESSED_" in variableName else ""
    DivideMassFlux_string = ("_DivideMassFluxMean" if "_DivideMassFluxMean" in variableName else
                             "_DivideMassFlux"     if "_DivideMassFlux"     in variableName else
                             "")

    if variableName in ["A_g","A_c","z","x","Z","Y","X","W","QCQI"]:
        dataType = "LagrangianArrays"
        dataName = "Lagrangian_Binary_Array"
        dataFolder = dataName
    elif variableName in  ["LFC","LCL"]:
        dataType = "LagrangianArrays"
        dataName = "LFC"       
        dataFolder = dataName
    elif variableName in [f"{Processed_string}A_g",f"{Processed_string}A_c"]:
        dataType = "LagrangianArrays"
        dataName = f"{Processed_string}Lagrangian_Binary_Array"
        dataFolder = dataName
    elif variableName in  ["QV", "QCQI",
                           "RH_vapor",
                           "VMF_g", "VMF_c",
                           "HMC", "CONVERGENCE",
                           "THETA","THETA_v", "THETA_e",
                           "BUOYANCY","BUOYANCY2",
                           "MSE","QC"]: #QC
        dataType = "LagrangianArrays"
        dataName = "VARS"       
        dataFolder = dataName
    elif variableName in ["QV_prime","QCQI_prime","RH_vapor_prime",
                           "W_prime","VMF_g_prime",'VMF_c_prime',
                           "HMC_prime",
                           "THETA_v_prime",'THETA_e_prime','MSE_prime']:
        dataType = "LagrangianArrays"
        dataName = "VARS_Perturbations"       
        dataFolder = dataName

    elif variableName in  ["UPDRAFT_AREA_g", "UPDRAFT_AREA_c",
                           "UPDRAFT_EDGE_DISTANCE_g","UPDRAFT_EDGE_DISTANCE_c"]:
        dataType = "LagrangianArrays"
        dataName = "UPDRAFT_AREA"       
        dataFolder = dataName
    
    elif variableName in ['E_g_eulerian', 'D_g_eulerian', 
                          'E_c_eulerian', 'D_c_eulerian',
                          'E_g_eulerian_DivideMassFlux', 'D_g_eulerian_DivideMassFlux', 
                          'E_c_eulerian_DivideMassFlux', 'D_c_eulerian_DivideMassFlux']:
        dataType = "LagrangianArrays"
        dataName = "Eulerian_Entrainment"
        dataFolder = "Eulerian_Entrainment"
    elif variableName in ['D_c', 'D_g', 'E_c', 'E_g',
                          'TransferD_c', 'TransferD_g', 
                          'TransferE_c', 'TransferE_g']:
        dataType = "LagrangianArrays"
        dataName = "Lagrangian_Entrainment"
        dataFolder = "Lagrangian_Entrainment"

    elif variableName in [f'{Processed_string}D{DivideMassFlux_string}_c', f'{Processed_string}D{DivideMassFlux_string}_g',
                          
                          f'{Processed_string}E{DivideMassFlux_string}_c', f'{Processed_string}E{DivideMassFlux_string}_g',
                          f'{Processed_string}TransferD{DivideMassFlux_string}_c', f'{Processed_string}TransferD{DivideMassFlux_string}_g', 
                          f'{Processed_string}TransferE{DivideMassFlux_string}_c', f'{Processed_string}TransferE{DivideMassFlux_string}_g']:
        dataType = "LagrangianArrays"
        dataName = f"{Processed_string}Lagrangian_Entrainment{DivideMassFlux_string}"
        dataFolder = f"{Processed_string}Lagrangian_Entrainment{DivideMassFlux_string}"

    elif variableName in ['WB_BUOY', 'WB_HADV', 'WB_HIDIFF', 'WB_HTURB',
                    'WB_PGRAD', 'WB_VADV', 'WB_VIDIFF', 'WB_VTURB']:
        dataType = "LagrangianArrays"
        dataName = "BUDGET_VARS_W"       
        dataFolder = "BUDGET_VARS"
    elif variableName in ['QVB_HADV', 'QVB_HIDIFF', 'QVB_HTURB', 'QVB_MP',
                    'QVB_VADV', 'QVB_VIDIFF', 'QVB_VTURB']:
        dataType = "LagrangianArrays"
        dataName = "BUDGET_VARS_QV"       
        dataFolder = "BUDGET_VARS"
    elif variableName in ['PTB_DISS', 'PTB_DIV', 'PTB_HADV', 'PTB_HIDIFF', 
                    'PTB_HTURB', 'PTB_MP', 'PTB_RAD', 'PTB_VADV', 
                    'PTB_VIDIFF', 'PTB_VTURB']:
        dataType = "LagrangianArrays"
        dataName = "BUDGET_VARS_TH"       
        dataFolder = "BUDGET_VARS"
        
    inputDataDirectory = os.path.normpath(
        os.path.join(DataManager.inputDirectory, "..", dataType,
                     f"Simulation_{ModelData.simulationNumber}_{DataManager.res}_{DataManager.t_res}_{DataManager.Nz_str}nz", dataFolder))

    if loadData:
        var_data = DataManager.GetTimestepData(inputDataDirectory, timeString,
                                               variableName=variableName, dataName=dataName,
                                               printstatement=printstatement)
        return var_data
    else:
        return inputDataDirectory,dataName

In [ ]:
dryMask = outputDictionary_Entrainment[f'{Processed_string}dryMask_c']
moistMask = outputDictionary_Entrainment[f'{Processed_string}moistMask_c']

In [ ]:
#loading more variables
QV = GetQCQI(t,varName='QV'); QV_Prior = GetQCQI(t-1,varName='QV')

W = GetQCQI(t,varName='W'); W_Prior = GetQCQI(t-1,varName='W')
a=ModelData.OpenParcel(); W=a['w'].isel(time=t); W_Prior=a['w'].isel(time=t-1)

QC = GetQCQI(t,varName='QC'); QC_Prior = GetQCQI(t-1,varName='QC')
RH = GetQCQI(t,varName='RH_vapor'); RH_Prior = GetQCQI(t-1,varName='RH_vapor')

Z = GetQCQI(t,varName='Z'); Z_Prior = GetQCQI(t-1,varName='Z')

In [ ]:
#calculating temperature
TH = GetQCQI(t,varName='THETA'); TH_Prior = GetQCQI(t-1,varName='THETA')

# a=ModelData.OpenData()
prs = a['prs'].isel(time=t).data.compute(); prs_Prior = a['prs'].isel(time=t-1).data.compute()
Y = GetQCQI(t,varName='Y'); Y_Prior = GetQCQI(t-1,varName='Y')
X = GetQCQI(t,varName='X'); X_Prior = GetQCQI(t-1,varName='X')
PRS=np.zeros_like(Z, dtype='float32'); PRS_Prior = PRS.copy()
PRS=prs[Z,Y,X];PRS_Prior=prs_Prior[Z,Y,X]

T = TH * (PRS/1e5)**0.286; T_Prior = TH_Prior * (PRS_Prior/1e5)**0.286

In [ ]:
#calculating rain
qr = a['qr'].isel(time=t).data.compute(); qr_Prior = a['qr'].isel(time=t-1).data.compute()
QR=np.zeros_like(Z, dtype='float32'); QR_Prior = QR.copy()
QR=qr[Z,Y,X];QR_Prior=qr_Prior[Z,Y,X]

In [ ]:
def GetVariables(mask): #mask=dryMask/moistMask
    qv_t=QV[mask]*1e3
    qv_tm1=QV_Prior[mask]*1e3
    
    qc_t=QC[mask]*1e3
    qc_tm1=QC_Prior[mask]*1e3

    qcqi_t=QCQI[mask]*1e3
    qcqi_tm1=QCQI_Prior[mask]*1e3
    
    rh_t=RH[mask]*1e2
    rh_tm1=RH_Prior[mask]*1e2

    t_t=T[mask]
    t_tm1=T_Prior[mask]
    
    w_t=W[mask]
    w_tm1=W_Prior[mask]

    qr_t=QR[mask]*1e3
    qr_tm1=QR_Prior[mask]*1e3

    z_t=ModelData.zh[Z[mask]]*1e3
    z_tm1=ModelData.zh[Z_Prior[mask]]*1e3

    return [qv_t,qv_tm1,
            qc_t,qc_tm1,
            qcqi_t,qcqi_tm1,
            rh_t,rh_tm1,
            t_t,t_tm1,
            w_t,w_tm1,
            qr_t,qr_tm1,
            z_t,z_tm1]
    
[qv_t_D,qv_tm1_D,
 qc_t_D,qc_tm1_D,
 qcqi_t_D,qcqi_tm1_D,
 rh_t_D,rh_tm1_D,
 t_t_D,t_tm1_D,
 w_t_D,w_tm1_D,
 qr_t_D,qr_tm1_D,
 z_t_D,z_tm1_D] = GetVariables(mask=dryMask)

[qv_t_M,qv_tm1_M,
 qc_t_M,qc_tm1_M,
 qcqi_t_M,qcqi_tm1_M,
 rh_t_M,rh_tm1_M,
 t_t_M,t_tm1_M,
 w_t_M,w_tm1_M,
 qr_t_M,qr_tm1_M,
 z_t_M,z_tm1_M] = GetVariables(mask=moistMask)

In [ ]:
#Preparing Data
data1 = qv_tm1_D - qv_t_D
data2 = qv_tm1_M - qv_t_M
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# Shared x-limits across both panels
xlo = min(np.percentile(data1, 1), np.percentile(data2, 1))
xhi = max(np.percentile(data1, 99), np.percentile(data2, 99))

# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")

for axis in axes:
    axis.set_xlabel('qv (g/kg)')
plt.suptitle(fr'$\phi_e - \phi_u$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

# qc_tm1_M[qv_tm1_M>qv_t_M].max()>1e-3 are any moist entrainment cases in cloud prior to entrainment ==> no

In [ ]:
#Preparing Data
data1 = qc_tm1_D; data2 = qc_tm1_M
# data1 = qcqi_tm1_D-qc_tm1_D; data2 = qcqi_tm1_M-qc_tm1_M
# data1=data1[data1<1e-3]; data2=data2[data2<1e-3]
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

xlo = min(np.percentile(data1, 5), np.percentile(data2, 5))
xhi = max(np.percentile(data1, 95), np.percentile(data2, 95))

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")
for axis in axes:
    axis.set_xlabel('qc (g/kg)')
plt.suptitle(fr'$\phi_e$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

apply_scientific_notation(axes.ravel())

In [ ]:
# #Preparing Data
# data1 = rh_tm1_D; data2 = rh_tm1_M
# data1=data1[(data1>=100)]; data2=data2[(data2>=100)]
# totalN = len(data1) + len(data2)
# weights1 = np.ones_like(data1) / totalN * 100
# weights2 = np.ones_like(data2) / totalN * 100

# xlo = min(np.percentile(data1, 5), np.percentile(data2, 5))
# xhi = max(np.percentile(data1, 95), np.percentile(data2, 95))

# #Plotting
# fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# # Dry
# axis=axes[0]
# axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
# axis.set_xlim(xlo, xhi)
# axis.set_title("Dry")
# # Moist
# axis=axes[1]
# axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
# axis.set_xlim(xlo, xhi)
# axis.set_title("Moist")
# for axis in axes:
#     axis.set_xlabel('rh (%) >= 100')
# plt.suptitle(fr'$\phi_e$')
# axes[0].set_ylabel(f'% / (dry + moist)')
# plt.tight_layout()

In [ ]:
#Preparing Data
data1 = t_tm1_D - t_t_D
data2 = t_tm1_M - t_t_M
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# Shared x-limits across both panels
xlo = min(np.percentile(data1, 1), np.percentile(data2, 1))
xhi = max(np.percentile(data1, 99), np.percentile(data2, 99))

# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")

for axis in axes:
    axis.set_xlabel('T (K)')
plt.suptitle(fr'$\phi_e - \phi_u$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

# qc_tm1_M[qv_tm1_M>qv_t_M].max()>1e-3 are any moist entrainment cases in cloud prior to entrainment ==> no

In [ ]:
#Preparing Data
data1 = w_tm1_D; data2 = w_tm1_M
# data1=data1[(data1>0.5)]; data2=data2[(data2>0.5)]
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

xlo = min(np.percentile(data1, 5), np.percentile(data2, 5))
xhi = max(np.percentile(data1, 95), np.percentile(data2, 95))

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")
for axis in axes:
    axis.set_xlabel('w (m/s)')
plt.suptitle(fr'$\phi_e$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

In [ ]:
#Preparing Data
data1 = qr_tm1_D; data2 = qr_tm1_M
# data1=data1[(data1<=1e-3)]; data2=data2[(data2<=1e-3)]
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

xlo = min(np.percentile(data1, 5), np.percentile(data2, 5))
xhi = max(np.percentile(data1, 95), np.percentile(data2, 95))

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")
for axis in axes:
    axis.set_xlabel('qr (g/kg)')
plt.suptitle(fr'$\phi_e$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

In [ ]:
#Preparing Data
data1 = z_tm1_D - z_t_D
data2 = z_tm1_M - z_t_M
totalN = len(data1) + len(data2)
weights1 = np.ones_like(data1) / totalN * 100
weights2 = np.ones_like(data2) / totalN * 100

#Plotting
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

# Shared x-limits across both panels
xlo = min(np.percentile(data1, 1), np.percentile(data2, 1))
xhi = max(np.percentile(data1, 99), np.percentile(data2, 99))

# Dry
axis=axes[0]
axis.hist(data1, bins=100, range=(xlo, xhi), weights=weights1)
axis.set_xlim(xlo, xhi)
axis.set_title("Dry")
# Moist
axis=axes[1]
axis.hist(data2, bins=100, range=(xlo, xhi), weights=weights2)
axis.set_xlim(xlo, xhi)
axis.set_title("Moist")

for axis in axes:
    axis.set_xlabel('z_grid (m)')
plt.suptitle(fr'$\phi_e - \phi_u$')
axes[0].set_ylabel(f'% / (dry + moist)')
plt.tight_layout()

# qc_tm1_M[qv_tm1_M>qv_t_M].max()>1e-3 are any moist entrainment cases in cloud prior to entrainment ==> no